# Quantum Entanglement Link -- Interactive Demo

A simulation-first quantum communication network stack in Python.
Density matrix formalism throughout, explicit noise everywhere.

## 1. Core Primitives

### Bell states, fidelity, purity, concurrence

In [ ]:
import numpy as np
from quantumnet.core import QubitState, apply, H, CNOT, X, Z, measure
from quantumnet.protocols import *

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

In [ ]:
phi_plus = QubitState.bell_phi_plus()
print(f'|Phi+> purity: {phi_plus.purity():.4f}')
print(f'|Phi+> concurrence: {phi_plus.concurrence():.4f}')
print(f'|Phi+> rho diagonal:\n{np.diag(phi_plus.rho)}')

In [ ]:
# Fidelity between Bell states
phi_minus = QubitState.bell_phi_minus()
psi_plus = QubitState.bell_psi_plus()
print(f'<Phi+|Phi-> = {phi_plus.fidelity(phi_minus):.4f}')
print(f'<Phi+|Psi+> = {phi_plus.fidelity(psi_plus):.4f}')

## 2. QKD Protocols

In [ ]:
print('=== BB84 QKD ===')
r = run_bb84(512, noise=0.01, rng=rng)
print(f'Key length: {len(r["key"])} bits')
print(f'QBER:       {r["qber"]:.4f}')
print(f'Key prefix: {r["key"][:32]}')

In [ ]:
print('=== E91 QKD ===')
r = run_e91(512, noise=0.01, rng=rng)
print(f'Key length: {len(r["key"])} bits')
print(f'QBER:       {r["qber"]:.4f}')
print(f'CHSH S:     {r["s_value"]:.4f} (violation if > 2)')
print(f'Key prefix: {r["key"][:32]}')

In [ ]:
# QBER vs noise scan
print('QBER vs depolarizing noise:')
for noise in [0.0, 0.02, 0.05, 0.1, 0.15]:
    r = run_bb84(1024, noise=noise, rng=rng)
    print(f'  noise={noise:.2f} -> QBER={r["qber"]:.4f}, key={len(r["key"])} bits')

## 3. Quantum Teleportation

In [ ]:
print('=== Teleportation ===')
r = run_teleportation(rng=rng)
print(f'Input state fidelity:  {r["input_fidelity"]:.6f}')
print(f'Teleported fidelity:   {r["teleported_fidelity"]:.6f}')
print(f'Bell outcome:          {r["bell_outcome"]}')
print(f'Success:               {r["success"]}')

## 4. Superdense Coding

In [ ]:
print('=== Superdense Coding ===')
for msg in range(4):
    r = run_superdense(bits=msg, rng=rng)
    ok = 'OK' if r['success'] else 'FAIL'
    print(f'  {msg:02b} -> {r["decoded_bits"]}  {ok}')

## 5. Entanglement Swapping

In [ ]:
print('=== Entanglement Swapping ===')
r = run_swapping(noise=0.0, rng=rng)
print(f'Noiseless: bell_outcome={r["bell_outcome"]}, fidelity={r["swapped_fidelity"]:.6f}')
for noise in [0.05, 0.1, 0.2]:
    r = run_swapping(noise=noise, rng=rng)
    print(f'  noise={noise}: fidelity={r["swapped_fidelity"]:.6f}')

## 6. Shor 9-Qubit Error Correction

In [ ]:
print('=== Shor 9-qubit Code ===')
for state_fn, label in [(QubitState.zero, 'zero'), (QubitState.plus, 'plus')]:
    orig = state_fn()
    enc = shor_encode(orig)
    print(f'{label}: encoded purity={enc.purity():.4f}')
    for qi in [0, 4, 8]:
        noisy = apply(X, enc, targets=[qi])
        corrected = shor_correct(noisy)
        dec = shor_decode(corrected)
        fid = orig.fidelity(dec)
        print(f'  X error on q{qi} -> corrected, fid={fid:.6f}')
        noisy = apply(Z, enc, targets=[qi])
        corrected = shor_correct(noisy)
        dec = shor_decode(corrected)
        fid = orig.fidelity(dec)
        print(f'  Z error on q{qi} -> corrected, fid={fid:.6f}')

## 7. Steane [[7,1,3]] Error Correction

In [ ]:
print('=== Steane 7-qubit Code ===')
for state_fn, label in [(QubitState.zero, 'zero'), (QubitState.plus, 'plus')]:
    orig = state_fn()
    enc = steane_encode(orig)
    print(f'{label}: encoded purity={enc.purity():.4f}')
    for qi in [0, 3, 6]:
        noisy = apply(X, enc, targets=[qi])
        corrected = steane_correct(noisy)
        dec = steane_decode(corrected)
        fid = orig.fidelity(dec)
        print(f'  X error on q{qi} -> corrected, fid={fid:.6f}')
        noisy = apply(Z, enc, targets=[qi])
        corrected = steane_correct(noisy)
        dec = steane_decode(corrected)
        fid = orig.fidelity(dec)
        print(f'  Z error on q{qi} -> corrected, fid={fid:.6f}')

## 8. Entanglement Distillation

In [ ]:
print('=== BBPSSW Distillation ===')
pairs = prepare_noisy_bell_pairs(10, fidelity=0.75, rng=rng)
print(f'Input pairs: {len(pairs)}, target fidelity=0.75')
for i in range(len(pairs) // 2):
    result = bbssw_distill(pairs[2*i], pairs[2*i+1], rng=rng)
    if result['success']:
        print(f'  Pair {i}: success, distilled fid={result["distilled_fidelity"]:.4f}')

In [ ]:
print('=== Deutsch Distillation ===')
pairs = prepare_noisy_bell_pairs(10, fidelity=0.75, rng=rng)
for i in range(len(pairs) // 2):
    result = deutsch_distill(pairs[2*i], pairs[2*i+1], rng=rng)
    if result['success']:
        print(f'  Pair {i}: success, distilled fid={result["distilled_fidelity"]:.4f}')

## 9. Quantum Memory Buffer (T1/T2 Decoherence)

In [ ]:
print('=== T1/T2 Memory ===')
state = QubitState.one()
print(f'Initial state: |1>, T1=10, T2=10')
for t in [0, 2, 5, 10, 20]:
    noisy = apply_t1_t2_noise(state, t=t, t1=10.0, t2=10.0)
    fid = state.fidelity(noisy)
    pop_0 = noisy.rho[0,0]
    print(f'  t={t:3d}: fidelity={fid:.4f}, population|0>={pop_0:.4f}')

In [ ]:
print('=== Cutoff Time ===')
state = QubitState.one()
t_cut = memory_cutoff_time(state, t1=10.0, t2=10.0, threshold=0.5)
print(f'Cutoff time (threshold=0.5): {t_cut:.4f}')
buf = QuantumMemoryBuffer(t1=10.0, t2=10.0, cutoff_fidelity=0.5)
buf.store('q1', state, current_time=0.0)
retrieved = buf.retrieve('q1', current_time=t_cut * 2)
print(f'Retrieve after 2x cutoff: {"dropped" if retrieved is None else "kept"}')

## 10. Full Pipeline: Encode -> Error -> Correct -> Decode

In [ ]:
print('=== Shor Code: Full Error Correction Pipeline ===')
n_errors = 0
n_corrected = 0
for state_fn, label in [(QubitState.zero, '0'), (QubitState.one, '1'), (QubitState.plus, '+')]:
    for qi in range(9):
        for gate, gname in [(X, 'X'), (Z, 'Z')]:
            orig = state_fn()
            enc = shor_encode(orig)
            noisy = apply(gate, enc, targets=[qi])
            corrected = shor_correct(noisy)
            dec = shor_decode(corrected)
            fid = orig.fidelity(dec)
            n_errors += 1
            if fid > 0.99:
                n_corrected += 1
            else:
                print(f'  FAIL: |{label}> {gname} on q{qi}: fid={fid:.4f}')
print(f'Corrected {n_corrected}/{n_errors} single-qubit errors')

In [ ]:
print('=== Steane Code: Full Error Correction Pipeline ===')
n_errors = 0
n_corrected = 0
for state_fn, label in [(QubitState.zero, '0'), (QubitState.one, '1'), (QubitState.plus, '+')]:
    for qi in range(7):
        for gate, gname in [(X, 'X'), (Z, 'Z')]:
            orig = state_fn()
            enc = steane_encode(orig)
            noisy = apply(gate, enc, targets=[qi])
            corrected = steane_correct(noisy)
            dec = steane_decode(corrected)
            fid = orig.fidelity(dec)
            n_errors += 1
            if fid > 0.99:
                n_corrected += 1
            else:
                print(f'  FAIL: |{label}> {gname} on q{qi}: fid={fid:.4f}')
print(f'Corrected {n_corrected}/{n_errors} single-qubit errors')

## Done

All protocols verified. For CLI usage: `python -m quantumnet all`

To run this notebook: `jupyter notebook notebooks/demo.ipynb`